# Simulation: Conditional Turning Bands

This notebook continues the geostatistical workflow from [3-estimation.ipynb](./3-estimation.ipynb). It uses the composites and variogram created in [1-populate-workspace.ipynb](./geostatistics/1-populate-workspace.ipynb) to run a **conditional simulation** with Seequent Evo Compute. Unlike estimation, simulation generates multiple plausible grade realisations, making it possible to explore spatial uncertainty.

The simulation writes summary statistics, quantiles, cutoff probabilities, and saved realisations to a **regular 3-D grid**, not directly to the estimation block model. Running this notebook creates a new variogram and grid in the selected workspace.

In [ ]:
import numpy as np
import plotly.express as px

import evo.compute.tasks as evo_tasks
import evo.objects.typed as evo_objs
from evo.compute.tasks.geostatistics.conditioned_simulator import BlockDiscretization, ConSimParameters
from evo.notebooks import ServiceManagerWidget

px.defaults.template = "simple_white"
%load_ext evo.widgets

## Authenticate and load the input objects

Select the workspace populated by the first geostatistics notebook. The input objects are references on Evo; their full datasets are not downloaded just to configure the compute task.

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(client_id="<CLIENT-ID>").login()

In [ ]:
comps = await evo_objs.object_from_path(evo_connection, "WP comps.json")
data_variogram = await evo_objs.object_from_path(evo_connection, "Cu_pct Variogram Model.json")
block_model = await evo_objs.object_from_path(evo_connection, "WP Regular BlockModel.json")

## Prepare a normal-score variogram

The variogram from the estimation notebook was defined in **data space** with sill 0.84. Conditional simulation requires a **normal-score** variogram with unit sill. For this demonstration, scale the nugget and structure contributions to sum to one while preserving their ranges and orientations. **This is an illustrative approximation, not a model fitted to normal-score-transformed samples**; fit and validate a normal-score variogram for production work.

In [ ]:
scale = data_variogram.sill
normal_score_variogram = await evo_objs.Variogram.create(
    evo_connection,
    evo_objs.VariogramData(
        name="Cu_pct NScore Variogram",
        description="Cu_pct NScore Variogram",
        sill=1.0,
        nugget=data_variogram.nugget / scale,
        is_rotation_fixed=data_variogram.is_rotation_fixed,
        modelling_space="normalscore",
        data_variance=1.0,
        attribute="Cu_pct",
        structures=[
            {**structure, "contribution": structure["contribution"] / scale} for structure in data_variogram.structures
        ],
    ),
)
normal_score_variogram

## Create the target grid and search neighbourhood

Conditional simulation targets a regular 3-D grid rather than a block-model reference. Use the existing block model's origin and extent, but double its cell size to keep the demonstration affordable. The ceiling on each grid dimension covers the full original extent. Confirm that the composites and block model use the same coordinate reference system (the populated WP examples use EPSG:32650).

In [ ]:
geometry = block_model.geometry
grid_cell_size = evo_objs.Size3d(
    dx=2 * geometry.block_size.dx,
    dy=2 * geometry.block_size.dy,
    dz=2 * geometry.block_size.dz,
)
grid_size = evo_objs.Size3i(
    nx=np.ceil(geometry.n_blocks.nx / 2),
    ny=np.ceil(geometry.n_blocks.ny / 2),
    nz=np.ceil(geometry.n_blocks.nz / 2),
)
grid = await evo_objs.Regular3DGrid.create(
    evo_connection,
    evo_objs.Regular3DGridData(
        name="WP Cu_pct Simulation Grid",
        description="Coarse target grid for conditional copper simulation",
        origin=geometry.origin,
        size=grid_size,
        cell_size=grid_cell_size,
        coordinate_reference_system=evo_objs.EpsgCode(32650),
    ),
)
grid

## Configure and run conditional simulation

Generate 20 conditional realisations using a fixed seed, saving five realisations to the grid. The summary attributes use all 20 realisations. P10, P50, P90 and the probability of exceeding 0.5% copper are computed per grid cell. Increasing the number of cells, realisations, lines, or sub-blocks increases compute cost.

In [ ]:
search = evo_tasks.SearchNeighborhood(
    ellipsoid=normal_score_variogram.get_ellipsoid().scaled(2.0), max_samples=24, min_samples=1
)

In [ ]:
simulation_params = ConSimParameters(
    source_object=comps,
    source_attribute=comps.attributes["Cu_pct"],
    target_object=grid,
    variogram_model=normal_score_variogram,
    neighborhood=search,
    kriging_method="simple",
    block_discretization=BlockDiscretization(nx=3, ny=3, nz=2),
    number_of_lines=500,
    number_of_simulations=20,
    number_of_simulations_to_save=5,
    random_seed=38239342,
    location_wise_quantiles=[0.1, 0.5, 0.9],
    probability_above_cutoff=[0.5],
)
simulation_params

In [ ]:
result = await evo_tasks.run(evo_connection, simulation_params)
result

## Inspect the results

The result exposes the *actual* attribute names returned by the service. Retrieve only the scalar summary, quantile, and cutoff columns for plotting; saved realisations are ensemble attributes and remain accessible on the target grid.

In [ ]:
print(f"Simulation target: {result.target_name}")
print(f"Mean attribute: {result.summary_attributes.mean.name}")
print(f"Variance attribute: {result.summary_attributes.variance.name}")
for quantile in result.quantile_attributes:
    print(f"P{quantile.quantile * 100:.0f}: {quantile.name}")
for cutoff in result.probability_above_cutoff_attributes:
    print(f"P(Cu > {cutoff.cutoff}%): {cutoff.name}")
if result.simulations_attribute:
    print(f"Saved realisations: {result.simulations_attribute.name}")

quantiles = {f"P{item.quantile * 100:.0f}": item.name for item in result.quantile_attributes}
scalar_columns = [
    result.summary_attributes.mean.name,
    result.summary_attributes.variance.name,
    *quantiles.values(),
    *(item.name for item in result.probability_above_cutoff_attributes),
]
simulation_df = await result.to_dataframe(*scalar_columns)
simulation_df.head()

In [ ]:
simulation_df[scalar_columns].describe()

## Visualise uncertainty across the grid

Each quantile is calculated **at a cell across realisations**. The histograms below show the distribution of those cell-wise quantiles across the grid; they are not histograms of individual realisations.

In [ ]:
quantile_df = (
    simulation_df[list(quantiles.values())]
    .rename(columns={column: label for label, column in quantiles.items()})
    .melt(var_name="Quantile", value_name="Cu (%)")
    .dropna()
)
fig = px.histogram(
    quantile_df,
    x="Cu (%)",
    color="Quantile",
    category_orders={"Quantile": list(quantiles)},
    barmode="overlay",
    opacity=0.65,
    nbins=50,
    histnorm="probability density",
    title="Conditional simulation: cell-wise copper quantiles",
)
fig.show()

In [ ]:
cutoff_attribute = result.probability_above_cutoff_attributes[0]
fig = px.histogram(
    simulation_df,
    x=cutoff_attribute.name,
    nbins=40,
    title=f"Probability of exceeding {cutoff_attribute.cutoff}% Cu by grid cell",
    labels={cutoff_attribute.name: "Probability"},
)
fig.show()

The simulation grid, its summaries, and its saved realisations are available in the selected Evo workspace. To publish these results into a block model, use a separate grid-to-block-model workflow; the compute task does not update the existing estimation block model.